In [ ]:
import speech_recognition as sr
import webbrowser
import pyttsx3
import pygame
import os
import time
import ollama
from gtts import gTTS

# -----------------------------
# Initialize modules
# -----------------------------
recognizer = sr.Recognizer()
engine = pyttsx3.init()

# Set your Ollama model here
MODEL_NAME = "llama3"   # Change to "phi3", "tinyllama", or "llama3.2" if needed


# -----------------------------
# Speak Function
# -----------------------------
def speak(text):
    """
    Speaks the given text using gTTS + pygame.
    If gTTS fails, fallback to pyttsx3.
    """
    print("Friday:", text)

    try:
        filename = "temp.mp3"

        tts = gTTS(text=text, lang="en")
        tts.save(filename)

        if not pygame.mixer.get_init():
            pygame.mixer.init()

        pygame.mixer.music.load(filename)
        pygame.mixer.music.play()

        while pygame.mixer.music.get_busy():
            pygame.time.Clock().tick(10)

        pygame.mixer.music.unload()
        time.sleep(0.2)

        if os.path.exists(filename):
            os.remove(filename)

    except Exception as e:
        print("Speech error:", e)
        engine.say(text)
        engine.runAndWait()


# -----------------------------
# AI Processing
# -----------------------------
def ai_process(command):
    """
    Sends user command to Ollama and returns short AI reply.
    """
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "You are Friday, a helpful AI assistant. Reply in one short sentence."
                },
                {
                    "role": "user",
                    "content": command
                }
            ]
        )
        return response["message"]["content"]

    except Exception as e:
        print("Ollama error:", e)
        return "Sorry, I could not connect to the AI model."


def get_news_headlines(topic="technology, world, science"):
    """
    Generates 5 short news headlines using Ollama.
    """
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "You are Friday, a helpful AI assistant. Give exactly 5 short news headlines only."
                },
                {
                    "role": "user",
                    "content": f"Give me 5 short latest headlines about {topic}, numbered 1 to 5."
                }
            ]
        )
        return response["message"]["content"]

    except Exception as e:
        print("News error:", e)
        return None


# -----------------------------
# Command Processing
# -----------------------------
def process_command(command):
    """
    Processes recognized voice commands.
    """
    command = command.lower().strip()

    if "open google" in command:
        speak("Opening Google")
        webbrowser.open("https://www.google.com")

    elif "open youtube" in command:
        speak("Opening YouTube")
        webbrowser.open("https://www.youtube.com")

    elif "open facebook" in command:
        speak("Opening Facebook")
        webbrowser.open("https://www.facebook.com")

    elif "open linkedin" in command:
        speak("Opening LinkedIn")
        webbrowser.open("https://www.linkedin.com")

    elif "news" in command:
        speak("Fetching news headlines")
        headlines = get_news_headlines()

        if headlines:
            for line in headlines.split("\n"):
                line = line.strip()
                if line:
                    speak(line)
        else:
            speak("Sorry, I could not get news right now.")

    else:
        speak("Let me think")
        reply = ai_process(command)
        speak(reply)


# -----------------------------
# Main Program
# -----------------------------
if __name__ == "__main__":
    speak("Initializing Friday")

    while True:
        print("Listening for wake word...")

        try:
            with sr.Microphone() as source:
                recognizer.adjust_for_ambient_noise(source, duration=0.5)
                audio = recognizer.listen(source, timeout=5, phrase_time_limit=3)

            word = recognizer.recognize_google(audio).lower().strip()
            print("Heard wake word:", word)

            if "friday" in word:
                speak("Yes")

                with sr.Microphone() as source:
                    print("Friday Active...")
                    recognizer.adjust_for_ambient_noise(source, duration=0.5)
                    audio = recognizer.listen(source, timeout=5, phrase_time_limit=8)

                command = recognizer.recognize_google(audio).lower().strip()
                print("Command:", command)

                if any(stop_word in command for stop_word in ["stop", "exit", "shutdown", "close"]):
                    print("Friday shutting down")
                    speak("Friday shutting down")
                    break

                process_command(command)

        except sr.WaitTimeoutError:
            continue

        except sr.UnknownValueError:
            continue

        except KeyboardInterrupt:
            print("\nProgram stopped by user.")
            speak("Goodbye")
            break

        except Exception as e:
            print("Error:", e)